<a href="https://colab.research.google.com/github/Cob-Lim/Table-Statistics-RAG/blob/main/RAG_Expenses_per_Country.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installing Modules

In [ ]:
!pip install -qU camelot-py[cv]
!pip install -qU tiktoken
!pip install -qU langchain-core langchain
print("Modules installed successfully")

Modules installed successfully


In [ ]:
# Dependencies for Camelot
!apt-get install -y -qq ghostscript
!apt-get install -y -qq python3-tk

print("Dependencies installed successfully")

Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 123630 files and directories currently installed.)
Preparing to unpack .../0-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../1-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package fonts-noto-mono.
Preparing to unpack .../2-fonts-noto-mono_20201225-1build1_all.deb ...
Unpacking fonts-noto-mono (20201225-1build1) ...
Selecting previously unselected package fonts-urw-base35.
Preparing to unpack .../3-fonts-urw-base35_20200910-1_all.deb ...
Unpacking fonts-urw-base35 (20200910-1) ...
Selecting previously unselected package libgs9-common.
Preparing to unpack .../4-libgs9-common_9.55.0~dfsg1-0ubuntu5.10_all.deb ...
Unpacking libgs9-common (9.55.0~dfsg1-0ubuntu5.10) ...
Selecting previously unselected package l

In [ ]:
# Check Ghostscript version
!gs --version

# Import Tkinter to verify
import tkinter
print("Tkinter installed successfully")

9.55.0
Tkinter installed successfully


### Input the API Keys

In [ ]:
# Inputting the API keys
import getpass
import os

print("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = getpass.getpass()
print("Enter your Pinecone API Key: ")
os.environ["PINECONE_API_KEY"] = getpass.getpass()

PINECONE_ENVIRONMENT = "us-east-1"
PINECONE_INDEX_NAME = "updated-expenses-country"

print("API keys inputted successfully.")

Enter your OpenAI API Key: 
··········
Enter your Pinecone API Key: 
··········
API keys inputted successfully.


### Uploading the File

In [ ]:
from google.colab import files

# Upload the file(s)
uploaded = files.upload()

Saving 員工國外出差要點 Sample Data.md to 員工國外出差要點 Sample Data.md
Saving 員工國外出差要點Sample Data.pdf to 員工國外出差要點Sample Data.pdf


### Extracting the Contents of the Table with Camelot

In [ ]:
import camelot

# Extracting the tables from the pdf
tables = camelot.read_pdf('員工國外出差要點Sample Data.pdf', pages = '2-3', flavor = 'stream')

# Checking the dimensions of the tables
print(f" Page 2: {tables[0]}")
print(f" Page 3: {tables[1]}")

# Checking the accuracy of the extraction
print(f"Page 2: {tables[0].parsing_report}")
print(f"Page 3: {tables[1].parsing_report}")

# Converting the extracted tables to dataframe for further processing
page_2_data = tables[0].df
page_3_data = tables[1].df

print("Tables converted to DataFrame successfully.")

 Page 2: <Table shape=(38, 7)>
 Page 3: <Table shape=(12, 7)>
Page 2: {'accuracy': 99.1, 'whitespace': 18.42, 'order': 1, 'page': 2}
Page 3: {'accuracy': 99.92, 'whitespace': 14.29, 'order': 1, 'page': 3}
Tables converted to DataFrame successfully.


In [ ]:
import pandas as pd

# Initialize an empty DataFrame
result_df = pd.DataFrame()

# Recursively combine all the tables into one
for i in range(len(tables)):
    df_i = tables[i].df
    result_df = pd.concat([result_df, df_i], axis = 0, ignore_index = True)

header_columns = ['國家', '總經理 - 住宿費', '總經理 - 餐費', '高階主管 - 住宿費', '高階主管 - 餐費', '一般人員 - 住宿費', '一般人員 - 餐費']

result_df.columns = header_columns

print("Tables combined successfully.")

result_df

Tables combined successfully.


,國家,總經理 - 住宿費,總經理 - 餐費,高階主管 - 住宿費,高階主管 - 餐費,一般人員 - 住宿費,一般人員 - 餐費
0,,,,,,,單位：美元/日
1,,,總經理,,高階主管,一般人員,
2,名稱(地區、國家、城市或其他),,,,,,
3,,住宿費,餐費,住宿費,餐費,住宿費 餐費,
4,亞洲地區,,,,,,
5,中國大陸,370,110,270,110,215,95
6,日本(Japan),445,140,345,140,275,105
7,南韓(Korea),415,130,315,130,255,110
8,菲律賓(Philippines),360,105,260,105,210,85
9,泰國(Thailand),445,100,245,100,200,85


In [ ]:
# Removing unnecessary rows
combined_data = result_df.iloc[4:]

# Place the value '中國大陸(China)' at row 5 of the first column
combined_data.at[5, combined_data.columns[0]] = '中國大陸(China)'

# Place the value '阿拉伯聯合大公國(United Arab Emirates)' at row 19 of the first column
combined_data.at[19, combined_data.columns[0]] = '阿拉伯聯合大公國(United Arab Emirates)'

# Delete rows 18 and 20
combined_data = combined_data.drop([18, 20]).reset_index(drop=True)

print("Initial cleaning successful.")

combined_data

Initial cleaning successful.


,國家,總經理 - 住宿費,總經理 - 餐費,高階主管 - 住宿費,高階主管 - 餐費,一般人員 - 住宿費,一般人員 - 餐費
0,亞洲地區,,,,,,
1,中國大陸(China),370,110,270,110,215,95
2,日本(Japan),445,140,345,140,275,105
3,南韓(Korea),415,130,315,130,255,110
4,菲律賓(Philippines),360,105,260,105,210,85
5,泰國(Thailand),445,100,245,100,200,85
6,馬來西亞(Malaysia),360,100,260,100,190,95
7,新加坡(Singapore),500,160,400,160,320,140
8,印尼(Indonesia),360,100,260,100,190,95
9,緬甸(Burma),360,100,260,100,190,95


### Using LLM to Convert to Markdown File

In [ ]:
!pip install -qU openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.5/389.5 kB 8.8 MB/s eta 0:00:00


In [ ]:
# Convert dataframe to CSV format to easily represent it in the prompt
data_csv = combined_data.to_csv(index = False)

print("Data converted to CSV successfully.")

data_csv

Data converted to CSV successfully.


'國家,總經理 - 住宿費,總經理 - 餐費,高階主管 - 住宿費,高階主管 - 餐費,一般人員 - 住宿費,一般人員 - 餐費\n亞洲地區,,,,,,\n中國大陸(China),370,110,270,110,215,95\n日本(Japan),445,140,345,140,275,105\n南韓(Korea),415,130,315,130,255,110\n菲律賓(Philippines),360,105,260,105,210,85\n泰國(Thailand),445,100,245,100,200,85\n馬來西亞(Malaysia),360,100,260,100,190,95\n新加坡(Singapore),500,160,400,160,320,140\n印尼(Indonesia),360,100,260,100,190,95\n緬甸(Burma),360,100,260,100,190,95\n印度(India),400,120,300,120,240,105\n柬埔寨(Cambodia),340,90,240,90,145,85\n寮國(Laos),340,90,240,90,135,85\n越南(Vietnam),340,90,240,90,175,85\n阿拉伯聯合大公國(United Arab Emirates),395,160,395,160,315,120\n以色列(Israel),495,160,395,160,315,120\n土耳其(Turkey),335,95,235,95,190,75\n俄羅斯(Russia),600,105,320,105,250,95\n香港(Hong Kong),450,140,350,140,280,125\n美洲地區,,,,,,\n加拿大(Canada),500,160,400,160,320,140\n美國(U.S.A.),400,180,440,180,355,135\n墨西哥(Mexico),585,155,385,155,305,115\n巴西(Brazil),445,100,245,100,195,90\n智利(Chile),435,135,335,135,265,100\n阿根廷(Argentina),570,150,370,150,295,130\n烏拉圭(Uruguay),440,

In [ ]:
# Markdown template to be filled in by the LLM
template = """
| 地區     | 國家     | 總經理 - 住宿費 | 總經理 - 餐費  | 高階主管 - 住宿費 | 高階主管 - 餐費 | 一般人員 - 住宿費 | 一般人員 - 餐費 |
| ------- | -------- | -------------- | ------------- | ---------------- | -------------- | --------------- | -------------- |
| 亞洲地區 | 中國大陸 (China) | 檢據實支   | 100或檢據實支 | 111          | 222            | 11           | 22             |
| 亞洲地區 | 日本 (Japan)   | 檢據實支   | 110或檢據實支 | 112          | 223            | 12           | 23             |
| 美洲地區 | 加拿大  (Canada) | 檢據實支   | 120或檢據實支 | 113          | 224            | 14           | 25             |
"""

print("Markdown template made successfully.")

Markdown template made successfully.


In [ ]:
# LLM Prompt
prompt = f"""
You are given a DataFrame with the following data in CSV format:

{data_csv}

And the following markdown template:

{template}

1) Please convert the given DataFrame to match the markdown format. Ensure you fill in any necessary rows as per the pattern of the template.
2) Use '檢據實支' for the '總經理 - 住宿費' column and format 'XXX 或檢據實支' for the '總經理 - 餐費' column, where XXX is the actual amount value of 總經理 - 餐費 in the given DataFrame.
3) Ensure the markdown output looks like the given template, but uses the data from the provided DataFrame.
4) Align all the columns neatly. Space them out well.
"""

print("LLM Prompt made successfully.")

LLM Prompt made successfully.


In [ ]:
from openai import OpenAI

# Running the LLM
client = OpenAI(
    api_key = os.environ.get("OPENAI_API_KEY"),  # This is the default and can be omitted
)

# Requesting LLM to process the data
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=2000
)

print("LLM processed successfully.")

LLM processed successfully.


In [ ]:
# Accessing the content
table_markdown = response.choices[0].message.content.strip()
# Printing the content
print(table_markdown)

Here's your DataFrame converted to the markdown format as per your specifications:

```markdown
| 地區     | 國家                          | 總經理 - 住宿費 | 總經理 - 餐費  | 高階主管 - 住宿費 | 高階主管 - 餐費 | 一般人員 - 住宿費 | 一般人員 - 餐費 |
| -------- | ---------------------------- | --------------- | ------------- | ---------------- | -------------- | --------------- | -------------- |
| 亞洲地區 | 中國大陸 (China)              | 檢據實支        | 110或檢據實支 | 270              | 110            | 215             | 95             |
| 亞洲地區 | 日本 (Japan)                  | 檢據實支        | 140或檢據實支 | 345              | 140            | 275             | 105            |
| 亞洲地區 | 南韓 (Korea)                  | 檢據實支        | 130或檢據實支 | 315              | 130            | 255             | 110            |
| 亞洲地區 | 菲律賓 (Philippines)          | 檢據實支        | 105或檢據實支 | 260              | 105            | 210             | 85             |
| 亞洲地區 | 泰國 (Thailand)               | 檢據實支        | 100或檢據實支 | 245              | 100            | 200

In [ ]:
# Save to a markdown file
with open("Expenses per Country.md", "w", encoding="utf-8") as f:
    f.write(table_markdown)

print("Markdown conversion completed by LLM and saved as a markdown file.")

Markdown conversion completed by LLM and saved as a markdown file.


### Combining the text with the table in the PDF

In [ ]:
!pip install -qU PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00


In [ ]:
import PyPDF2

# Open the PDF file
with open('/content/員工國外出差要點Sample Data.pdf', 'rb') as file:
    reader = PyPDF2.PdfFileReader(file)
    first_page_text = ''

    # We only get page 1 as the other pages are already taken care of as tables
    page = reader.getPage(0)
    first_page_text += page.extract_text()
    text_markdown = first_page_text.strip()

print("Text extracted from the PDF successfully.")
print("\n")
print(text_markdown)

Text extracted from the PDF successfully.


員工國外出差要點    
  
為使本單位員工赴國外出差、考察或受訓等有所依循，特訂定本要點。   
  
1. (餐雜宿費標準 ) 員工跨境出差之宿費及餐雜費悉依附表一標準支給之。第四
條(飛機艙別及其他交通工具搭乘標準及交通費 ) 奉派出國人員往返機票費及
其他交通工具費用准予按實報支，其標準如附表二。   
2. 在國外必須之交通，應依經奉准旅程表之路程為之，其交通費須檢附收據按實報支，自辦公  
(住宿)處往返國際機場且來回車資，副總經理 (含)以上實報實銷，協理級 (含)以下同仁當次出
差往返國內外機場之費用總計上限新台幣 三千元得檢據核銷。   
3. (投保標準 )  
奉派出國人員得以出國公假日數為保險期間，投保子 單位旅遊保險中之意外 (傷害)死殘保險、傷
害醫療保險、海外突發疾病醫療 (含燒燙傷 )保險、海外旅遊不便保險，其標準如附表三。   
4. (其他費用 )  
辦理出國必備之包括護照費、簽證費、機場服務費等，得檢據請領；若於出差國家轉搭國內線航
班，其票價未含行李費者，該行李費得列入手續費，惟以 1件或20公斤為限。其他未列於本辦法
中費用，如因業務需要，以簽准內容為限，實報實銷。奉派出國人員交際應酬費用除奉准由 單位
開支之部份外，一概自理。   
5. (海外境內差旅費支給標準 ) 派駐至海外人員於工作地國家境內因公出差時，適用本條之規定。
交通費以往返出差地點所必須之交通費用為限，如附表四。海外境內出差之宿費及雜費，如附
表五。  
   
6. (費用核銷 )  
奉派出國人員由國外返回任所後，應於十天內填具出差費用報告單，並檢附相關收據，報請支付
旅費。  
7. (修訂及施行 )  
本要點之訂定、修正或廢止應經總經理同意。   
本要點自訂定日起生效施行，修正或廢止時亦同。


In [ ]:
# Combining the text and the table to be a complete markdown file
complete_markdown_text = text_markdown + "\n\n" + table_markdown

print("Complete Markdown text created successfully.")
print("\n")
print(complete_markdown_text)

員工國外出差要點    
  
為使本單位員工赴國外出差、考察或受訓等有所依循，特訂定本要點。   
  
1. (餐雜宿費標準 ) 員工跨境出差之宿費及餐雜費悉依附表一標準支給之。第四
條(飛機艙別及其他交通工具搭乘標準及交通費 ) 奉派出國人員往返機票費及
其他交通工具費用准予按實報支，其標準如附表二。   
2. 在國外必須之交通，應依經奉准旅程表之路程為之，其交通費須檢附收據按實報支，自辦公  
(住宿)處往返國際機場且來回車資，副總經理 (含)以上實報實銷，協理級 (含)以下同仁當次出
差往返國內外機場之費用總計上限新台幣 三千元得檢據核銷。   
3. (投保標準 )  
奉派出國人員得以出國公假日數為保險期間，投保子 單位旅遊保險中之意外 (傷害)死殘保險、傷
害醫療保險、海外突發疾病醫療 (含燒燙傷 )保險、海外旅遊不便保險，其標準如附表三。   
4. (其他費用 )  
辦理出國必備之包括護照費、簽證費、機場服務費等，得檢據請領；若於出差國家轉搭國內線航
班，其票價未含行李費者，該行李費得列入手續費，惟以 1件或20公斤為限。其他未列於本辦法
中費用，如因業務需要，以簽准內容為限，實報實銷。奉派出國人員交際應酬費用除奉准由 單位
開支之部份外，一概自理。   
5. (海外境內差旅費支給標準 ) 派駐至海外人員於工作地國家境內因公出差時，適用本條之規定。
交通費以往返出差地點所必須之交通費用為限，如附表四。海外境內出差之宿費及雜費，如附
表五。  
   
6. (費用核銷 )  
奉派出國人員由國外返回任所後，應於十天內填具出差費用報告單，並檢附相關收據，報請支付
旅費。  
7. (修訂及施行 )  
本要點之訂定、修正或廢止應經總經理同意。   
本要點自訂定日起生效施行，修正或廢止時亦同。

Here's your DataFrame converted to the markdown format as per your specifications:

```markdown
| 地區     | 國家                          | 總經理 - 住宿費 | 總經理 - 餐費  | 高階主管 - 住宿費 | 高階主管 - 餐費 | 一般人員 - 住宿費 | 一般人員 - 餐費 |
| -------- | ---

In [ ]:
# Save to a markdown file
with open("員工國外出差要點 Sample Data.md", "w", encoding="utf-8") as f:
    f.write(complete_markdown_text)

print("Markdown conversion completed and saved as a markdown file.")

Markdown conversion completed and saved as a markdown file.


## Main LLM Pipeline

*   Embedding model: OpenAI text-embedding-ada-002
*   Vector Database: Pinecone
*   LLM: OpenAI gpt-4o (can handle more complex tasks)

### Loading and Converting the File into Chunks

In [ ]:
# Load the Markdown file
file_path = '/content/員工國外出差要點 Sample Data.md'
with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

In [ ]:
print(content)

員工國外出差要點    
  
為使本單位員工赴國外出差、考察或受訓等有所依循，特訂定本要點。   
  
1. (餐雜宿費標準 ) 員工跨境出差之宿費及餐雜費悉依附表一標準支給之。第四
條(飛機艙別及其他交通工具搭乘標準及交通費 ) 奉派出國人員往返機票費及
其他交通工具費用准予按實報支，其標準如附表二。   
2. 在國外必須之交通，應依經奉准旅程表之路程為之，其交通費須檢附收據按實報支，自辦公  
(住宿)處往返國際機場且來回車資，副總經理 (含)以上實報實銷，協理級 (含)以下同仁當次出
差往返國內外機場之費用總計上限新台幣 三千元得檢據核銷。   
3. (投保標準 )  
奉派出國人員得以出國公假日數為保險期間，投保子 單位旅遊保險中之意外 (傷害)死殘保險、傷
害醫療保險、海外突發疾病醫療 (含燒燙傷 )保險、海外旅遊不便保險，其標準如附表三。   
4. (其他費用 )  
辦理出國必備之包括護照費、簽證費、機場服務費等，得檢據請領；若於出差國家轉搭國內線航
班，其票價未含行李費者，該行李費得列入手續費，惟以 1件或20公斤為限。其他未列於本辦法
中費用，如因業務需要，以簽准內容為限，實報實銷。奉派出國人員交際應酬費用除奉准由 單位
開支之部份外，一概自理。   
5. (海外境內差旅費支給標準 ) 派駐至海外人員於工作地國家境內因公出差時，適用本條之規定。
交通費以往返出差地點所必須之交通費用為限，如附表四。海外境內出差之宿費及雜費，如附
表五。  
   
6. (費用核銷 )  
奉派出國人員由國外返回任所後，應於十天內填具出差費用報告單，並檢附相關收據，報請支付
旅費。  
7. (修訂及施行 )  
本要點之訂定、修正或廢止應經總經理同意。   
本要點自訂定日起生效施行，修正或廢止時亦同。

Here's your DataFrame converted to the markdown format as per your specifications:

```markdown
| 地區     | 國家                          | 總經理 - 住宿費 | 總經理 - 餐費  | 高階主管 - 住宿費 | 高階主管 - 餐費 | 一般人員 - 住宿費 | 一般人員 - 餐費 |
| -------- | ---

In [ ]:
import re
import pandas as pd

# Parse the Markdown table
def parse_markdown_table(content):

    # Split the content into lines
    lines = content.splitlines()

    # Strip each line and drop empty ones
    lines = [line.strip() for line in lines if line.strip()]

    # Extract lines with table content
    table_lines = [line for line in lines if '|' in line]

    # Extract headers and rows
    headers = None
    rows = []
    for line in table_lines:
        if re.match(r'^\|.+\|$', line):  # Lines surrounded by pipes
            row = [cell.strip() for cell in line.split('|')[1:-1]]
            if headers is None:  # First row is the header
                headers = row
            else:  # Other rows are data
                rows.append(row)

    # Drop the first row in rows since it is a template
    rows = rows[1:]

    return headers, rows

# Extract headers and rows
headers, rows = parse_markdown_table(content)

# Convert to a DataFrame
final_df = pd.DataFrame(rows, columns = headers)

final_df

,地區,國家,總經理 - 住宿費,總經理 - 餐費,高階主管 - 住宿費,高階主管 - 餐費,一般人員 - 住宿費,一般人員 - 餐費
0,亞洲地區,中國大陸 (China),檢據實支,110或檢據實支,270,110,215,95
1,亞洲地區,日本 (Japan),檢據實支,140或檢據實支,345,140,275,105
2,亞洲地區,南韓 (Korea),檢據實支,130或檢據實支,315,130,255,110
3,亞洲地區,菲律賓 (Philippines),檢據實支,105或檢據實支,260,105,210,85
4,亞洲地區,泰國 (Thailand),檢據實支,100或檢據實支,245,100,200,85
5,亞洲地區,馬來西亞 (Malaysia),檢據實支,100或檢據實支,260,100,190,95
6,亞洲地區,新加坡 (Singapore),檢據實支,160或檢據實支,400,160,320,140
7,亞洲地區,印尼 (Indonesia),檢據實支,100或檢據實支,260,100,190,95
8,亞洲地區,緬甸 (Burma),檢據實支,100或檢據實支,260,100,190,95
9,亞洲地區,印度 (India),檢據實支,120或檢據實支,300,120,240,105


In [ ]:
# Chunk data into rows, while keeping the headers
chunks = []
text_for_embed = []

for index, row in final_df.iterrows():
    chunk = {col: row[col] for col in final_df.columns}
    chunks.append(chunk)

    # Serialize the chunk into text
    text = "; ".join([f"{key}: {value}" for key, value in chunk.items()])
    text_for_embed.append(text)

# Display chunks and their serialized text
for i, (chunk, text) in enumerate(zip(chunks, text_for_embed)):
    print(f"Chunk {i+1}:")
    for key, value in chunk.items():
        print(f"  {key}: {value}")
    print(f"Serialized Text: {text}")
    print()

Chunk 1:
  地區: 亞洲地區
  國家: 中國大陸 (China)
  總經理 - 住宿費: 檢據實支
  總經理 - 餐費: 110或檢據實支
  高階主管 - 住宿費: 270
  高階主管 - 餐費: 110
  一般人員 - 住宿費: 215
  一般人員 - 餐費: 95
Serialized Text: 地區: 亞洲地區; 國家: 中國大陸 (China); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 110或檢據實支; 高階主管 - 住宿費: 270; 高階主管 - 餐費: 110; 一般人員 - 住宿費: 215; 一般人員 - 餐費: 95

Chunk 2:
  地區: 亞洲地區
  國家: 日本 (Japan)
  總經理 - 住宿費: 檢據實支
  總經理 - 餐費: 140或檢據實支
  高階主管 - 住宿費: 345
  高階主管 - 餐費: 140
  一般人員 - 住宿費: 275
  一般人員 - 餐費: 105
Serialized Text: 地區: 亞洲地區; 國家: 日本 (Japan); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 140或檢據實支; 高階主管 - 住宿費: 345; 高階主管 - 餐費: 140; 一般人員 - 住宿費: 275; 一般人員 - 餐費: 105

Chunk 3:
  地區: 亞洲地區
  國家: 南韓 (Korea)
  總經理 - 住宿費: 檢據實支
  總經理 - 餐費: 130或檢據實支
  高階主管 - 住宿費: 315
  高階主管 - 餐費: 130
  一般人員 - 住宿費: 255
  一般人員 - 餐費: 110
Serialized Text: 地區: 亞洲地區; 國家: 南韓 (Korea); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 130或檢據實支; 高階主管 - 住宿費: 315; 高階主管 - 餐費: 130; 一般人員 - 住宿費: 255; 一般人員 - 餐費: 110

Chunk 4:
  地區: 亞洲地區
  國家: 菲律賓 (Philippines)
  總經理 - 住宿費: 檢據實支
  總經理 - 餐費: 105或檢據實支
  高階主管 - 住宿費: 260
  高階主管 - 餐費: 105
  一般

In [ ]:
text_for_embed

['地區: 亞洲地區; 國家: 中國大陸 (China); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 110或檢據實支; 高階主管 - 住宿費: 270; 高階主管 - 餐費: 110; 一般人員 - 住宿費: 215; 一般人員 - 餐費: 95',
 '地區: 亞洲地區; 國家: 日本 (Japan); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 140或檢據實支; 高階主管 - 住宿費: 345; 高階主管 - 餐費: 140; 一般人員 - 住宿費: 275; 一般人員 - 餐費: 105',
 '地區: 亞洲地區; 國家: 南韓 (Korea); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 130或檢據實支; 高階主管 - 住宿費: 315; 高階主管 - 餐費: 130; 一般人員 - 住宿費: 255; 一般人員 - 餐費: 110',
 '地區: 亞洲地區; 國家: 菲律賓 (Philippines); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 105或檢據實支; 高階主管 - 住宿費: 260; 高階主管 - 餐費: 105; 一般人員 - 住宿費: 210; 一般人員 - 餐費: 85',
 '地區: 亞洲地區; 國家: 泰國 (Thailand); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 100或檢據實支; 高階主管 - 住宿費: 245; 高階主管 - 餐費: 100; 一般人員 - 住宿費: 200; 一般人員 - 餐費: 85',
 '地區: 亞洲地區; 國家: 馬來西亞 (Malaysia); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 100或檢據實支; 高階主管 - 住宿費: 260; 高階主管 - 餐費: 100; 一般人員 - 住宿費: 190; 一般人員 - 餐費: 95',
 '地區: 亞洲地區; 國家: 新加坡 (Singapore); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 160或檢據實支; 高階主管 - 住宿費: 400; 高階主管 - 餐費: 160; 一般人員 - 住宿費: 320; 一般人員 - 餐費: 140',
 '地區: 亞洲地區; 國家: 印尼 (Indonesia); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費:

### Beginning the LLM Pipeline

In [ ]:
!pip install -qU langchain-openai langchain-pinecone pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 7.4 MB/s eta 0:00:00


In [ ]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone
pc = Pinecone(api_key = os.environ["PINECONE_API_KEY"], environment = PINECONE_ENVIRONMENT)

index = pc.Index(PINECONE_INDEX_NAME)

print("Pinecone initialized successfully.")

Pinecone initialized successfully.


In [ ]:
# Converting to embeddings and storing in vector database
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # Transforms text to vectors via embeddings
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

# List of unique IDs for each text
doc_ids = [f"text-{i}" for i in range(len(text_for_embed))]

# Add texts to the vector database
vector_store.add_texts(text_for_embed, ids = doc_ids)

print("Embeddings created successfully.")

Embeddings created successfully.


In [ ]:
# Initialize the LLM GPT-4o from OpenAI
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name='gpt-4o', temperature=0.1)

print("LLM initialized successfully.")

LLM initialized successfully.


In [ ]:
from langchain.prompts import ChatPromptTemplate

# Define a ChatPromptTemplate for interacting with the LLM
system_prompt = (
    """
        {context}

        --------------------------------------------------------------------------------------------------------------------------------------

        Based on the context provided above, answer the following question in the best way possible. Note that all amounts are in United States Dollars (USD) per day.

        Please ensure to:

          1. If calculations are required, provide the final answer without showing the detailed steps. For example, when calculating an average, simply state: "各國的平均值為 Y（實際平均值）。"
          2. Round off all answers to two decimal places.
          3. List the names of the countries involved in the final answer.
          4. Use complete sentences to explain your answer (not just numbers).

        Respond in traditional Chinese.

        Question: {input}

        Answer:
    """
)

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

print("ChatPromptTemplate initialized successfully.")

ChatPromptTemplate initialized successfully.


In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create the question-answering chain using the LLM and prompt
combine_docs_chain = create_stuff_documents_chain(llm, prompt_template)

# Create the Retrieval chain combining the retriever and QA chain
retriever = vector_store.as_retriever(search_kwargs={"k": 20})
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RetrievalQA chain created successfully.")

RetrievalQA chain created successfully.


### Topical Guardrail

In [ ]:
from langchain.prompts import ChatPromptTemplate

# Define a ChatPromptTemplate for the guardrail

guardrail_prompt = (

"""
Your task is to determine if a user query is permitted. Follow these rules:

1. If the query contains the role "總經理" alone, respond with "not_allowed."
2. If "總經理" appears along with other roles (e.g., "高階主管" or "一般人員"), modify the query to remove "總經理".
3. For all other queries, simply respond with "allowed."
"""

)
guardrail_template = ChatPromptTemplate.from_messages(
    [
        ("system", guardrail_prompt),
        ("human", "{input}"),
    ]
)

print("Guardrail Prompt Template initialized successfully.")

Guardrail Prompt Template initialized successfully.


In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Create a RunnableSequence, which is the new way to sequence prompts with LLMs
guardrail_chain = guardrail_template | llm | StrOutputParser()

print("Guardrail chain created successfully.")

Guardrail chain created successfully.


In [ ]:
guardrail_chain.invoke("在菲律賓，一位總經理在住宿和餐飲方面花費多少美元嗎？")

'not_allowed'

In [ ]:
guardrail_chain.invoke("How much does a general manager spend on accommodation and meals in the Philippines?")

'not_allowed'

In [ ]:
guardrail_chain.invoke("在菲律賓，一位最高的主管在住宿和餐飲方面花費多少美元嗎？")

'allowed'

In [ ]:
guardrail_chain.invoke("How much does the highest manager spend on accommodation and meals in the Philippines?")

'allowed'

In [ ]:
guardrail_chain.invoke("在南非共和國，一位總經理和一位一般人員在住宿和餐飲方面花費多少美元嗎？")

'allowed'

### Adding a Topical Guard Rail on 總經理

In [ ]:
!pip install -qU asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 6.7 MB/s eta 0:00:00


In [ ]:
import asyncio

# Async functions to use the LangChain LLMChain
async def get_chat_response(query):
    print("Getting LLM response...")
    response = await asyncio.to_thread(rag_chain.invoke, {"input": query})
    print("Got LLM response.")
    return response


async def topical_guardrail(query):
    print("Checking topical guardrail...")
    response = await asyncio.to_thread(guardrail_chain.invoke, {"input": query})
    print("Got guardrail response.")
    return response

In [ ]:
import asyncio

async def execute_chat_with_guardrail(query):

    # Start both tasks concurrently
    topical_guardrail_task = asyncio.create_task(topical_guardrail(query))
    chat_task = asyncio.create_task(get_chat_response(query))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when = asyncio.FIRST_COMPLETED
        )

        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response.strip().lower() == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered.")
                return "抱歉，我無法提供任何有關總經理的資訊。", None

            elif chat_task in done:
                chat_response = chat_task.result()
                answer = chat_response["answer"]
                context = chat_response["context"]
                return answer, context

        else:
            await asyncio.sleep(0.1)  # Sleep for a bit before checking the tasks again


# Synchronous entry point for querying
def ask_question(query):
    loop = asyncio.get_event_loop()
    if loop.is_running():
        # Handle the case where the event loop is already running
        return loop.create_task(execute_chat_with_guardrail(query))
    else:
        return loop.run_until_complete(execute_chat_with_guardrail(query))

### Query Questions

In [ ]:
# Example usage
simple_question = "在菲律賓，一位總經理在住宿和餐飲方面花費多少美元嗎？"
medium_question = "一位高階主管在香港住5天的住宿和餐飲費用是多少美元？"
semi_hard_question = "在南非共和國，一位總經理和一位一般人員在住宿和餐飲方面花費多少美元嗎？"
hard_question = "在美洲地區的平均總經理的住宿費是多少？確保包括數據中所有相關的國家。"
very_hard_question = "在歐洲地區的平均住宿費是多少？確保包括數據中所有相關的國家。首先按職位細分費用：總經理，高階主管，一般人員。"
extremely_hard_question = "在亞洲地區的平均餐飲費是多少？確保包括數據中所有相關的國家。首先按職位細分費用：總經理，高階主管，一般人員。"

answer, context = await ask_question(medium_question)

print("Answer:", answer)
print("\n")

if context == None:
  print("Sources: 没有資訊。")
else:
  for source in context:
      print("Sources:")
      print(source.page_content)

Checking topical guardrail
Getting LLM response
Got guardrail response
Got LLM response
Answer: 一位高階主管在香港住5天的住宿和餐飲費用為：住宿費每天350美元，餐費每天140美元。總共的費用為 (350 + 140) * 5 = 2450 美元。


Sources:
地區: 亞洲地區; 國家: 香港 (Hong Kong); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 140或檢據實支; 高階主管 - 住宿費: 350; 高階主管 - 餐費: 140; 一般人員 - 住宿費: 280; 一般人員 - 餐費: 125
Sources:
地區: 美洲地區; 國家: 美國 (U.S.A.); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 180或檢據實支; 高階主管 - 住宿費: 440; 高階主管 - 餐費: 180; 一般人員 - 住宿費: 355; 一般人員 - 餐費: 135
Sources:
地區: 亞洲地區; 國家: 中國大陸 (China); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 110或檢據實支; 高階主管 - 住宿費: 270; 高階主管 - 餐費: 110; 一般人員 - 住宿費: 215; 一般人員 - 餐費: 95
Sources:
地區: 歐洲地區; 國家: 盧森堡 (Luxembourg); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 190或檢據實支; 高階主管 - 住宿費: 470; 高階主管 - 餐費: 190; 一般人員 - 住宿費: 380; 一般人員 - 餐費: 165
Sources:
地區: 美洲地區; 國家: 加拿大 (Canada); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 160或檢據實支; 高階主管 - 住宿費: 400; 高階主管 - 餐費: 160; 一般人員 - 住宿費: 320; 一般人員 - 餐費: 140
Sources:
地區: 亞洲地區; 國家: 新加坡 (Singapore); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 160或檢據實支; 高階主管 - 住宿費: 400; 高階主管 - 餐費: 160; 一般人員 - 住宿費: 320;

In [ ]:
answer, context = await ask_question("在菲律賓，一位最高的主管在住宿和餐飲方面花費多少美元嗎？")

print("Answer:", answer)
print("\n")

if context == None:
  print("Sources: 没有資訊。")
else:
  for source in context:
      print("Sources:")
      print(source.page_content)

Checking topical guardrail
Getting LLM response
Got guardrail response
Got LLM response
Answer: 在菲律賓，一位最高的主管（即總經理）在住宿方面的花費為檢據實支，而在餐飲方面的花費為105美元或檢據實支。因此，總經理在菲律賓的餐飲花費至少為105美元，而住宿費則依據實際支出來決定。


Sources:
地區: 亞洲地區; 國家: 菲律賓 (Philippines); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 105或檢據實支; 高階主管 - 住宿費: 260; 高階主管 - 餐費: 105; 一般人員 - 住宿費: 210; 一般人員 - 餐費: 85
Sources:
地區: 亞洲地區; 國家: 馬來西亞 (Malaysia); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 100或檢據實支; 高階主管 - 住宿費: 260; 高階主管 - 餐費: 100; 一般人員 - 住宿費: 190; 一般人員 - 餐費: 95
Sources:
地區: 亞洲地區; 國家: 印尼 (Indonesia); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 100或檢據實支; 高階主管 - 住宿費: 260; 高階主管 - 餐費: 100; 一般人員 - 住宿費: 190; 一般人員 - 餐費: 95
Sources:
地區: 亞洲地區; 國家: 新加坡 (Singapore); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 160或檢據實支; 高階主管 - 住宿費: 400; 高階主管 - 餐費: 160; 一般人員 - 住宿費: 320; 一般人員 - 餐費: 140
Sources:
地區: 美洲地區; 國家: 美國 (U.S.A.); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 180或檢據實支; 高階主管 - 住宿費: 440; 高階主管 - 餐費: 180; 一般人員 - 住宿費: 355; 一般人員 - 餐費: 135
Sources:
地區: 亞洲地區; 國家: 香港 (Hong Kong); 總經理 - 住宿費: 檢據實支; 總經理 - 餐費: 140或檢據實支; 高階主管 - 住宿費: 350; 高階主管 - 餐費: 1

## Evaluation Pipeline

In [ ]:
!pip install -qU ragas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
queries = [
    "在菲律賓，一位總經理在住宿和餐飲方面花費多少美元嗎？",
    "在巴西，一位高階主管在住宿和餐飲方面花費多少美元嗎？",
    "在澳大利亞，一位一般人員在住宿和餐飲方面花費多少美元嗎？",
    "一位高階主管在香港住5天的住宿和餐飲費用是多少美元？",
    "一位總經理在紐西蘭住7天的住宿和餐飲費用是多少美元？",
    "一位一般人員在日本住10天的住宿和餐飲費用是多少美元？",
    "在南非共和國，一位總經理和一位一般人員在住宿和餐飲方面花費多少美元嗎？",
    "在加拿大，一位高階主管和一位一般人員在住宿和餐飲方面花費多少美元嗎？",
    "在美洲地區的平均總經理的住宿費是多少？確保包括數據中所有相關的國家。",
    "在歐洲地區的平均住宿費是多少？確保包括數據中所有相關的國家。首先按職位細分費用：總經理，高階主管，一般人員。",
]

In [ ]:
real_answers = [
    "抱歉，我無法提供任何有關總經理的資訊。",
    "在巴西，一位高階主管每天在住宿上花費245美元，餐飲花費100美元，共計每天在住宿和餐飲上花費345美元。",
    "在澳大利亞，一位一般人員每天在住宿上花費265美元，餐飲花費115美元，共計每天在住宿和餐飲上花費380美元。",
    "在香港，一位高階主管在5天的住宿上花費1,750美元，餐飲花費700美元，共計5天的住宿和餐飲花費2,450美元。",
    "抱歉，我無法提供任何有關總經理的資訊。",
    "在日本，一位一般人員在10天的住宿上花費2,750美元，餐飲花費1,050美元，共計5天的住宿和餐飲花費3,800美元。",
    "在南非共和國，一位一般人員每天在住宿上花費205美元，餐飲花費90美元，共計每天在住宿和餐飲上花費295美元。",
    "在加拿大，一位高階主管每天在住宿上花費400美元，餐飲花費160美元，共計每天在住宿和餐飲上花費560美元。此外，一位一般人員每天在住宿上花費320美元，餐飲花費140美元，共計每天在住宿和餐飲上花費460美元。這些人花費的總金額是1,020美元。",
    "抱歉，我無法提供任何有關總經理的資訊。",
    "在歐洲（包括法國、德國、荷蘭、比利時、盧森堡、瑞士、丹麥、瑞典、英國、西班牙、義大利），高階主管的平均餐飲費用為397.27美元，一般人員的平均餐飲費用為318.18美元。",
]

In [ ]:
import time
from tqdm import tqdm

# Store all answers of the model to the queries
model_answers = []

# Inner progress bar for queries for each model
for query in tqdm(queries, desc=f"Processing Queries", unit="query", leave=False):
    print("\n")
    try:
        # Gets the answer and the context
        answer, context = await ask_question(query)

        # Cleaning up the context
        if context == None:
            context = 'Information is confidential.'
        else:
            context = [doc.page_content for doc in context]

        # Appending the answers to the dictionary
        model_answers.append({
            'query': query,
            'answer': answer,
            'context': context
        })

    except Exception as e:

        model_answers.append({
            'query': query,
            'answer': f"Error: {e}",
            'context': ''
        })

    # Add sleep after each query to prevent rate limit issues
    time.sleep(3)

print("LLM has finished answering all queries.")

Processing Queries:   0%|          | 0/10 [00:00<?, ?query/s]

Checking topical guardrail
Getting LLM response
Got guardrail response
Topical guardrail triggered.


Processing Queries:  10%|█         | 1/10 [00:03<00:29,  3.26s/query]

Checking topical guardrail
Getting LLM response
Got guardrail response
Got LLM response


Processing Queries:  20%|██        | 2/10 [00:08<00:33,  4.16s/query]

...

Processing Queries:  80%|████████  | 8/10 [00:36<00:10,  5.01s/query]

Checking topical guardrail
Getting LLM response
Got guardrail response
Topical guardrail triggered.


Processing Queries:  90%|█████████ | 9/10 [00:40<00:04,  4.47s/query]

Checking topical guardrail
Getting LLM response
Got guardrail response
Got LLM response


LLM has finished answering all queries.


In [ ]:
import json

# Save the lists to their own text files
with open("Updated 員工國外出差要點 Sample Data (Responses).json", "w", encoding = "utf-8") as f:
    json.dump(model_answers, f, ensure_ascii = False, indent = 4)

In [ ]:
import json

# Load the list from the JSON file
with open("Updated 員工國外出差要點 Sample Data (Responses).json", "r") as f:
    model_answers = json.load(f)

In [ ]:
# Preparing datasets for evaluation
model_evaluation = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

for i in range(len(model_answers)):
    model_evaluation["question"].append(model_answers[i]['query'])
    model_evaluation["answer"].append(model_answers[i]['answer'])
    model_evaluation["contexts"].append(model_answers[i]['context'])
    model_evaluation["ground_truth"].append(real_answers[i])

# Flatten the nested lists by joining them and wrapping them back in a list
model_evaluation['contexts'] = [[' '.join(context)] if isinstance(context, list) else [context] for context in model_evaluation['contexts']]

In [ ]:
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from datasets import Dataset

# Create a dataset from the data
evaluation_dataset = Dataset.from_dict(model_evaluation)

# Define the metrics you want to compute
metrics = [context_precision, context_recall, faithfulness, answer_relevancy]

# Evaluate the datasets
evaluation_result = evaluate(
    dataset = evaluation_dataset,
    metrics = metrics,
    raise_exceptions = False
    )


print(f"Evaluation scores for Updated 員工國外出差要點 Sample Data: {evaluation_result}")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluation scores for Updated 員工國外出差要點 Sample Data: {'context_precision': 1.0000, 'context_recall': 0.8667, 'faithfulness': 0.7856, 'answer_relevancy': 0.6335}
